# Sioux Falls implementation of the coupled flow--structure equilibrium framework

**Reproducibility notebook for _Urban Transportation Economics Under Planetary Boundaries_.**

This notebook implements the Sioux Falls application used in the paper, including:

- geographic peripheral/central user classes;
- four commodities (user class × ICE/EV technology);
- fixed and elastic OD demand;
- BPR congestion with pseudo-lengths and speed-dependent ICE tailpipe emissions;
- accessibility floors evaluated at fleet-weighted generalized OD costs;
- the endogenous EV structural response and stationary share $s^*$;
- the equilibrium-admissibility threshold $s_E$;
- price-supported admissibility thresholds;
- emissions pricing with fixed versus endogenous structure;
- the sensitivity and robustness calculations reported in the paper.

The application is an illustrative benchmark, not an empirical model of Sioux Falls. The pseudo-lengths preserve internal consistency between travel time, speed, operating costs and emissions but should not be interpreted as geographical road lengths.

> **Units.** Time is in minutes, money in dollars, distance in km, and emissions in grams of CO$_2$. Use-stage emissions prices are stored internally in dollars per gram and reported as dollars per tonne by multiplying by $10^6$.


## 1. Setup and Sioux Falls data

In [ ]:
from pathlib import Path
import sys, math, json, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from dataclasses import replace

pd.set_option("display.precision", 6)
warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

BASE = "https://raw.githubusercontent.com/bstabler/TransportationNetworks/master/SiouxFalls"
files = {
    "SiouxFalls_net.tntp": f"{BASE}/SiouxFalls_net.tntp",
    "SiouxFalls_trips.tntp": f"{BASE}/SiouxFalls_trips.tntp",
    "SiouxFalls_node.tntp": f"{BASE}/SiouxFalls_node.tntp",
}
for name, url in files.items():
    path = DATA_DIR / name
    if not path.exists():
        print(f"Downloading {name} ...")
        urllib.request.urlretrieve(url, path)

print("Data ready:", sorted(p.name for p in DATA_DIR.glob("SiouxFalls_*.tntp")))

## 2. Core model

The next three cells contain the revised implementation used for the paper. They are written to local Python modules so the notebook remains self-contained while keeping the numerical code identical to the verified implementation.

One legacy naming detail is retained in the code: `tau0` and `tau1` are the internal names for the baseline and gap-indexed **purchase-stage** charge parameters $(\phi_0,\phi_1)$. The separate argument `tau` is the **use-stage** emissions price on ICE travel.

In [ ]:
from pathlib import Path
Path('sf_model.py').write_text('"""\nSioux Falls implementation of the coupled flow–structure equilibrium framework\n(revised). Implements Section 4 of the revised manuscript:\n\n  * four-commodity (class × vehicle-type) user equilibrium with elastic demand,\n    solved as the convex potential problem of Prop. 4.1 (VOT-rescaled costs),\n    by Frank–Wolfe with exact line search and the proper FW duality gap;\n  * tailpipe emissions from ICE commodity flows; gravity accessibility at the\n    OD costs travellers face;\n  * vehicle-choice logit structural response with a gap-indexed carbon price;\n  * structural residual H = s − ŝ (sign per §4.5).\n\nUnits: time in minutes, money in $, distance in km, emissions in g CO2.\n"""\nfrom __future__ import annotations\nimport re\nfrom dataclasses import dataclass, field, replace\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix\nfrom scipy.sparse.csgraph import dijkstra\n\n# ----------------------------------------------------------------------------\n# Parameters (all documented in parameters.md with sources)\n# ----------------------------------------------------------------------------\n@dataclass\nclass Params:\n    # network / units\n    vff_kmh: float = 56.3                 # benchmark free-flow speed (35 mph) for pseudo-lengths\n    v_clip: tuple = (10.0, 90.0)          # speed clip (upper bound is unreachable: v <= vff)\n    # user classes\n    class_scheme: str = "geographic"      # "legacy" (paper) or "geographic" (revision)\n    share_P: float = 0.5                  # legacy only: share of every OD demand in class P\n    vot: dict = field(default_factory=lambda: {"P": 10.0/60, "C": 25.0/60})   # $/min\n    # vehicle operating cost $/km\n    oc: dict = field(default_factory=lambda: {"I": 0.170, "E": 0.085})\n    # emissions factor g/km: b0 + b1/v + b2 (v - vopt)^2\n    ef_b: tuple = (120.0, 900.0, 0.08)\n    v_opt: float = 50.0\n    # accessibility decay (1/$) and floors (fraction of baseline)\n    beta: dict = field(default_factory=lambda: {"P": 0.22, "C": 0.12})\n    floor_frac: dict = field(default_factory=lambda: {"P": 0.95, "C": 0.95})\n    # demand elasticity w.r.t. generalised OD cost (0 = fixed demand)\n    eps: float = 0.0\n    # ecological ceiling as fraction of baseline emissions\n    ebar_frac: float = 0.8\n    # vehicle choice (per-trip $ units): s_hat = sigma(theta*(Delta - kappa + tau*ebar_trip))\n    theta: float = 1.0        # 1/$ per trip\n    kappa: float = 1.0        # $ per trip (annualised EV capital premium / trips per year)\n    tau0: float = 0.0         # $/g baseline emissions price\n    tau1: float = 0.0         # $/g per unit normalised gap (gap-indexed component)\n    # solver\n    fw_tol: float = 1e-5\n    fw_max_iter: int = 2000\n\nCLASSES = ("P", "C")\nVTYPES = ("I", "E")\nCOMMODITIES = [(g, v) for g in CLASSES for v in VTYPES]\n\ndef sigmoid(u):\n    return 1.0 / (1.0 + np.exp(-u))\n\n# ----------------------------------------------------------------------------\n# Data\n# ----------------------------------------------------------------------------\ndef load_tntp_net(path):\n    rows = []\n    with open(path) as f:\n        for line in f:\n            line = line.strip()\n            if not line or line.startswith("<") or line.startswith("~"):\n                continue\n            parts = [p for p in re.split(r"\\s+", line.replace(";", "")) if p]\n            if len(parts) < 8:\n                continue\n            tail, head, cap, length, fft, b, power = (int(parts[0]), int(parts[1]), float(parts[2]),\n                                                      float(parts[3]), float(parts[4]), float(parts[5]), float(parts[6]))\n            rows.append(dict(tail=tail, head=head, cap=cap, length_raw=length, fft=fft, b=b, power=power))\n    df = pd.DataFrame(rows).reset_index(drop=True)\n    df["idx"] = np.arange(len(df))\n    return df\n\ndef load_tntp_trips(path):\n    od = {}\n    origin = None\n    with open(path) as f:\n        for line in f:\n            line = line.strip()\n            if not line or line.startswith("<"):\n                continue\n            if line.lower().startswith("origin"):\n                origin = int(line.split()[1]); continue\n            for item in line.split(";"):\n                item = item.strip()\n                if not item or ":" not in item:\n                    continue\n                d, q = item.split(":")\n                d, q = int(d), float(q)\n                if q > 0 and d != origin:\n                    od[(origin, d)] = q\n    return od\n\ndef load_tntp_nodes(path):\n    rows = []\n    with open(path) as f:\n        for line in f:\n            parts = [p for p in re.split(r"\\s+", line.strip().replace(";", "")) if p]\n            if len(parts) >= 3 and parts[0].isdigit():\n                rows.append(dict(node=int(parts[0]), x=float(parts[1]), y=float(parts[2])))\n    return pd.DataFrame(rows)\n\n# ----------------------------------------------------------------------------\n# Network object\n# ----------------------------------------------------------------------------\nclass Network:\n    def __init__(self, links: pd.DataFrame, nodes: pd.DataFrame, od: dict, prm: Params):\n        self.prm = prm\n        self.links = links.copy()\n        self.n = int(max(links["tail"].max(), links["head"].max()))\n        self.tail = links["tail"].values - 1\n        self.head = links["head"].values - 1\n        self.cap = links.cap.values.astype(float)\n        self.a0 = links.fft.values.astype(float)                       # minutes\n        self.power = links.power.values\n        self.bcoef = links.b.values\n        # polynomial form t = a0 + a4 z^4  (power = 4 for all Sioux Falls links)\n        self.a4 = self.a0 * self.bcoef / self.cap ** self.power\n        self.L = prm.vff_kmh * self.a0 / 60.0                          # pseudo-length km\n        self.A = len(links)\n        self.link_index = {(t, h): i for i, (t, h) in enumerate(zip(self.tail, self.head))}\n        # node groups by distance to centroid\n        nodes = nodes.sort_values("node").reset_index(drop=True)\n        cx, cy = nodes.x.mean(), nodes.y.mean()\n        kx = np.cos(np.deg2rad(cy))                                    # project lon/lat to approx. planar\n        dist = np.hypot((nodes.x - cx) * kx, nodes.y - cy)\n        nodes["dist"] = dist\n        if prm.class_scheme == "legacy":\n            q75, q25 = dist.quantile(0.75), dist.quantile(0.25)\n            self.group_nodes = {"P": set(nodes.node[dist >= q75]), "C": set(nodes.node[dist <= q25])}\n        else:  # geographic: outer half vs inner half\n            med = dist.median()\n            self.group_nodes = {"P": set(nodes.node[dist > med]), "C": set(nodes.node[dist <= med])}\n        # class potential demand q̄_{g,w}\n        self.od_pairs = sorted(od.keys())\n        self.W = len(self.od_pairs)\n        self.od_o = np.array([o for o, d in self.od_pairs]) - 1\n        self.od_d = np.array([d for o, d in self.od_pairs]) - 1\n        q_all = np.array([od[w] for w in self.od_pairs])\n        self.qbar = {}\n        if prm.class_scheme == "legacy":\n            self.qbar["P"] = prm.share_P * q_all\n            self.qbar["C"] = (1 - prm.share_P) * q_all\n        else:\n            inP = np.array([o in self.group_nodes["P"] for o, d in self.od_pairs])\n            self.qbar["P"] = np.where(inP, q_all, 0.0)\n            self.qbar["C"] = np.where(~inP, q_all, 0.0)\n        self.q_total = q_all.sum()\n        # destination opportunity weights\n        self.W_dest = np.zeros(self.n)\n        for (o, d), q in od.items():\n            self.W_dest[d - 1] += q\n        # reference OD costs (set by calibrate_reference)\n        self.pibar = {g: None for g in CLASSES}\n        self.E0 = None\n        self.A0 = None\n\n    # ---- link-level functions -------------------------------------------\n    def travel_time(self, z):\n        return self.a0 + self.a4 * np.maximum(z, 0.0) ** 4\n\n    def speed(self, z):\n        v = self.L / np.maximum(self.travel_time(z), 1e-9) * 60.0     # km/h\n        return np.clip(v, *self.prm.v_clip)\n\n    def ef(self, v):\n        b0, b1, b2 = self.prm.ef_b\n        return b0 + b1 / v + b2 * (v - self.prm.v_opt) ** 2\n\n    def e_link(self, z):\n        """per-ICE-vehicle emissions on each link, g/veh"""\n        return self.L * self.ef(self.speed(z))\n\n    def link_cost(self, z, g, v, tau=0.0):\n        """generalised link cost in $ for commodity (g,v)"""\n        c = self.prm.vot[g] * self.travel_time(z) + self.L * self.prm.oc[v]\n        if v == "I" and tau > 0:\n            c = c + tau * self.e_link(z)\n        return c\n\n    # ---- shortest paths ---------------------------------------------------\n    def _graph(self, w):\n        return csr_matrix((w, (self.tail, self.head)), shape=(self.n, self.n))\n\n    def shortest_paths(self, w):\n        """Dijkstra from all nodes; returns (dist matrix, predecessor matrix)."""\n        G = self._graph(np.maximum(w, 1e-12))\n        D, P = dijkstra(G, directed=True, return_predecessors=True)\n        return D, P\n\n    def aon(self, w, q_od, DP=None):\n        """All-or-nothing load of OD demands q_od (array over od_pairs) on costs w.\n        Returns link flows and OD min costs. DP = precomputed (dist, pred)."""\n        D, P = self.shortest_paths(w) if DP is None else DP\n        y = np.zeros(self.A)\n        pi = D[self.od_o, self.od_d]\n        for k in range(self.W):\n            q = q_od[k]\n            if q <= 0:\n                continue\n            o, d = self.od_o[k], self.od_d[k]\n            node = d\n            while node != o:\n                prev = P[o, node]\n                y[self.link_index[(prev, node)]] += q\n                node = prev\n        return y, pi\n\n    # ---- demand functions ---------------------------------------------------\n    def rho(self, s, v):\n        return (1.0 - s) if v == "I" else s\n\n    def demand(self, g, v, s, pi):\n        """q_{(g,v),w} = rho_v(s) * qbar_g,w * (pi/pibar_g,w)^(-eps)"""\n        base = self.rho(s, v) * self.qbar[g]\n        if self.prm.eps <= 0 or self.pibar[g] is None:\n            return base\n        ratio = np.where(base > 0, np.maximum(pi, 1e-9) / self.pibar[g], 1.0)\n        return base * ratio ** (-self.prm.eps)\n\n    def inv_demand(self, g, v, s, q):\n        """D̂^{-1}: cost that induces commodity demand q (money $). Only where base>0."""\n        base = self.rho(s, v) * self.qbar[g]\n        out = np.full_like(q, np.inf)\n        m = base > 0\n        if self.prm.eps <= 0:\n            out[m] = self.pibar[g][m] if self.pibar[g] is not None else 0.0\n            return out\n        out[m] = self.pibar[g][m] * (np.maximum(q[m], 1e-12) / base[m]) ** (-1.0 / self.prm.eps)\n        return out\n\n# ----------------------------------------------------------------------------\n# Equilibrium solver (Prop. 4.1: potential problem, Frank–Wolfe + line search)\n# ----------------------------------------------------------------------------\n@dataclass\nclass EqResult:\n    s: float\n    tau: float\n    x: dict            # commodity link flows\n    q: dict            # commodity OD demands\n    pi: dict           # commodity OD min costs ($)\n    z: np.ndarray      # aggregate link flow\n    gap: float\n    iters: int\n    converged: bool\n\ndef solve_ue(net: Network, s: float, tau: float = 0.0, warm: EqResult | None = None,\n             tol=None, max_iter=None, verbose=False) -> EqResult:\n    prm = net.prm\n    tol = prm.fw_tol if tol is None else tol\n    max_iter = prm.fw_max_iter if max_iter is None else max_iter\n    vot = prm.vot\n\n    def costs_all(z):\n        return {(g, v): net.link_cost(z, g, v, tau) for (g, v) in COMMODITIES}\n\n    def aon_all(z, q_cur=None):\n        """direction point: AON at current costs with demand D(pi) (elastic) or fixed."""\n        c = costs_all(z)\n        y, qy, pi = {}, {}, {}\n        for (g, v) in COMMODITIES:\n            # OD costs on this commodity\'s graph\n            DP = net.shortest_paths(np.maximum(c[(g, v)], 1e-12))\n            pi_m = DP[0][net.od_o, net.od_d]\n            q_m = net.demand(g, v, s, pi_m)\n            y_m, _ = net.aon(c[(g, v)], q_m, DP)\n            y[(g, v)], qy[(g, v)], pi[(g, v)] = y_m, q_m, pi_m\n        return y, qy, pi, c\n\n    # initialisation\n    if warm is not None and abs(warm.s - s) < 0.5:\n        x = {k: warm.x[k].copy() for k in COMMODITIES}\n        q = {k: warm.q[k].copy() for k in COMMODITIES}\n        # rescale flows/demands to the current split so they are feasible\n        for (g, v) in COMMODITIES:\n            r_old, r_new = net.rho(warm.s, v), net.rho(s, v)\n            f = (r_new / r_old) if r_old > 0 else 0.0\n            x[(g, v)] *= f; q[(g, v)] *= f\n            if r_old == 0 and r_new > 0:\n                q[(g, v)] = r_new * net.qbar[g]\n                x[(g, v)], _ = net.aon(net.link_cost(np.zeros(net.A), g, v, tau), q[(g, v)])\n    else:\n        z0 = np.zeros(net.A)\n        y, qy, pi, c = aon_all(z0)\n        x = {k: y[k].copy() for k in COMMODITIES}\n        q = {k: qy[k].copy() for k in COMMODITIES}\n\n    def aggregate(x):\n        return sum(x[k] for k in COMMODITIES)\n\n    gap = np.inf; it = 0; converged = False\n    for it in range(1, max_iter + 1):\n        z = aggregate(x)\n        y, qy, pi, c = aon_all(z)\n        # FW duality gap in rescaled (time) units: <grad Phi, w - y>\n        dz = aggregate(y) - z\n        g_flow = float(np.dot(net.travel_time(z), -dz))\n        g_lin = 0.0; g_dem = 0.0; norm = float(np.dot(net.travel_time(z), z))\n        for (g, v) in COMMODITIES:\n            kappa = (c[(g, v)] - vot[g] * net.travel_time(z)) / vot[g]     # rescaled monetary part\n            g_lin += float(np.dot(kappa, x[(g, v)] - y[(g, v)]))\n            norm += float(np.dot(kappa, x[(g, v)]))\n            if prm.eps > 0:\n                dinv = net.inv_demand(g, v, s, q[(g, v)]) / vot[g]\n                m = np.isfinite(dinv)\n                g_dem += float(np.dot(-dinv[m], q[(g, v)][m] - qy[(g, v)][m]))\n        gap_abs = g_flow + g_lin + g_dem\n        gap = gap_abs / max(abs(norm), 1e-9)\n        if verbose and (it == 1 or it % 25 == 0):\n            print(f"  FW it {it:4d}  rel.gap {gap:.3e}")\n        if gap < tol:\n            converged = True\n            break\n        # exact line search on alpha in [0,1]: bisection on dPhi/dalpha (convex)\n        dx = {k: y[k] - x[k] for k in COMMODITIES}\n        dq = {k: qy[k] - q[k] for k in COMMODITIES}\n        def dphi(al):\n            za = z + al * dz\n            val = float(np.dot(net.travel_time(za), dz))\n            for (g, v) in COMMODITIES:\n                kappa = (net.link_cost(za, g, v, tau) - vot[g] * net.travel_time(za)) / vot[g]\n                val += float(np.dot(kappa, dx[(g, v)]))\n                if prm.eps > 0:\n                    qa = q[(g, v)] + al * dq[(g, v)]\n                    dinv = net.inv_demand(g, v, s, qa) / vot[g]\n                    m = np.isfinite(dinv)\n                    val -= float(np.dot(dinv[m], dq[(g, v)][m]))\n            return val\n        lo, hi = 0.0, 1.0\n        if dphi(1.0) <= 0:\n            al = 1.0\n        elif dphi(0.0) >= 0:\n            al = 0.0\n        else:\n            for _ in range(40):\n                mid = 0.5 * (lo + hi)\n                if dphi(mid) > 0: hi = mid\n                else: lo = mid\n            al = 0.5 * (lo + hi)\n        for k in COMMODITIES:\n            x[k] = x[k] + al * dx[k]\n            q[k] = q[k] + al * dq[k]\n    z = aggregate(x)\n    _, _, pi, _ = aon_all(z)\n    return EqResult(s=s, tau=tau, x=x, q=q, pi=pi, z=z, gap=gap, iters=it, converged=converged)\n\n# ----------------------------------------------------------------------------\n# Evaluation at an equilibrium\n# ----------------------------------------------------------------------------\ndef emissions(net: Network, eq: EqResult) -> float:\n    z_ice = eq.x[("P", "I")] + eq.x[("C", "I")]\n    return float(np.dot(z_ice, net.e_link(eq.z)))\n\ndef class_od_cost(net: Network, eq: EqResult, g: str):\n    """fleet-weighted OD cost faced by a class-g traveller ($)."""\n    s = eq.s\n    return (1 - s) * eq.pi[(g, "I")] + s * eq.pi[(g, "E")]\n\ndef accessibility(net: Network, eq: EqResult, g: str) -> float:\n    """gravity accessibility of group g at the OD costs travellers face."""\n    pi_g = class_od_cost(net, eq, g)\n    beta = net.prm.beta[g]\n    origins = net.group_nodes[g]\n    vals = []\n    # need all destinations incl. those with zero demand: use dijkstra on class cost graph\n    c_mix = (1 - eq.s) * net.link_cost(eq.z, g, "I", eq.tau) + eq.s * net.link_cost(eq.z, g, "E", eq.tau)\n    D, _ = net.shortest_paths(np.maximum(c_mix, 1e-12))\n    for o in origins:\n        d_row = D[o - 1]\n        acc = float(np.sum(net.W_dest * np.exp(-beta * d_row) * np.isfinite(d_row)))\n        vals.append(acc)\n    return float(np.mean(vals))\n\ndef ice_trip_emissions(net: Network, eq: EqResult) -> float:\n    """average g CO2 per ICE trip (used in vehicle choice)."""\n    trips = float(eq.q[("P", "I")].sum() + eq.q[("C", "I")].sum())\n    return emissions(net, eq) / trips if trips > 0 else 0.0\n\ndef ev_cost_saving(net: Network, eq: EqResult) -> float:\n    """trip-weighted generalised-cost advantage of EV over ICE, $ per trip (Delta)."""\n    num = 0.0; den = 0.0\n    for g in CLASSES:\n        w = net.qbar[g]\n        num += float(np.dot(w, eq.pi[(g, "I")] - eq.pi[(g, "E")])); den += float(w.sum())\n    return num / den\n\ndef s_hat(net: Network, eq: EqResult, E_bar: float) -> tuple[float, float]:\n    """vehicle-choice desired EV share with gap-indexed price. Returns (s_hat, tau_eff)."""\n    prm = net.prm\n    E = emissions(net, eq)\n    tau_eff = prm.tau0 + prm.tau1 * (E - E_bar) / net.E0\n    tau_eff = max(tau_eff, 0.0)\n    Delta = ev_cost_saving(net, eq)\n    ebar = ice_trip_emissions(net, eq)\n    u = prm.theta * (Delta - prm.kappa + tau_eff * ebar)\n    return float(sigmoid(u)), float(tau_eff)\n\ndef boundary(net: Network, eq: EqResult, E_bar: float, floors: dict) -> dict:\n    E = emissions(net, eq)\n    A = {g: accessibility(net, eq, g) for g in CLASSES}\n    B = max(E - E_bar, max(floors[g] - A[g] for g in CLASSES))\n    return dict(E=E, A=A, B=B, eco_ok=E <= E_bar, soc_ok=all(A[g] >= floors[g] for g in CLASSES))\n\n# ----------------------------------------------------------------------------\n# Baseline / reference calibration\n# ----------------------------------------------------------------------------\ndef calibrate_reference(net: Network, verbose=True) -> EqResult:\n    """Solve the fixed-demand baseline (s=0, tau=0), set reference OD costs pibar,\n    baseline emissions E0 and accessibilities A0."""\n    eps_saved = net.prm.eps\n    net.prm.eps = 0.0\n    eq0 = solve_ue(net, s=0.0, tau=0.0, verbose=verbose)\n    net.prm.eps = eps_saved\n    for g in CLASSES:\n        net.pibar[g] = np.maximum(eq0.pi[(g, "I")], 1e-9)\n    net.E0 = emissions(net, eq0)\n    net.A0 = {g: accessibility(net, eq0, g) for g in CLASSES}\n    return eq0\n\ndef build(prm: Params, data_dir="data") -> Network:\n    links = load_tntp_net(f"{data_dir}/SiouxFalls_net.tntp")\n    od = load_tntp_trips(f"{data_dir}/SiouxFalls_trips.tntp")\n    nodes = load_tntp_nodes(f"{data_dir}/SiouxFalls_node.tntp")\n    return Network(links, nodes, od, prm)\n')
print('wrote sf_model.py')

In [ ]:
from pathlib import Path
Path('analysis.py').write_text('"""Thresholds, residual, and sweeps on top of sf_model (Sections 4.3–4.7)."""\nfrom __future__ import annotations\nimport numpy as np, pandas as pd\nfrom scipy.optimize import brentq\nfrom sf_model import (Params, build, calibrate_reference, solve_ue, emissions, accessibility,\n                      s_hat, boundary, ev_cost_saving, ice_trip_emissions, CLASSES, EqResult)\n\nclass Model:\n    def __init__(self, prm: Params, data_dir="data", verbose=False):\n        self.prm = prm\n        self.net = build(prm, data_dir)\n        self.eq0 = calibrate_reference(self.net, verbose=verbose)\n        self.E0 = self.net.E0\n        self.A0 = self.net.A0\n        self.E_bar = prm.ebar_frac * self.E0\n        self.floors = {g: prm.floor_frac[g] * self.A0[g] for g in CLASSES}\n        self._cache: dict[tuple, EqResult] = {}\n\n    # ---- equilibrium with caching and warm start --------------------------\n    def eq(self, s: float, tau: float = 0.0) -> EqResult:\n        key = (round(s, 8), round(tau, 10))\n        if key in self._cache:\n            return self._cache[key]\n        warm = None\n        cands = [k for k in self._cache if abs(k[1] - key[1]) < 1e-12]\n        if cands:\n            kw = min(cands, key=lambda k: abs(k[0] - key[0]))\n            warm = self._cache[kw]\n        r = solve_ue(self.net, s, tau, warm=warm)\n        self._cache[key] = r\n        return r\n\n    def point(self, s: float, tau: float = 0.0, E_bar=None) -> dict:\n        E_bar = self.E_bar if E_bar is None else E_bar\n        r = self.eq(s, tau)\n        b = boundary(self.net, r, E_bar, self.floors)\n        sh, tau_eff = s_hat(self.net, r, E_bar)\n        return dict(s=s, tau=tau, E=b["E"], E_frac=b["E"] / self.E0, A_P=b["A"]["P"], A_C=b["A"]["C"],\n                    B=b["B"], eco_ok=b["eco_ok"], soc_ok=b["soc_ok"], s_hat=sh, R=sh - s, tau_eff=tau_eff,\n                    Delta=ev_cost_saving(self.net, r), ebar_trip=ice_trip_emissions(self.net, r),\n                    q_total=sum(r.q[k].sum() for k in r.q), gap=r.gap, iters=r.iters)\n\n    # ---- thresholds ---------------------------------------------------------\n    def _root(self, f, lo=0.0, hi=1.0, xtol=1e-5):\n        flo, fhi = f(lo), f(hi)\n        if flo <= 0: return lo, "at_lower_bound"\n        if fhi > 0: return np.nan, "none_in_[0,1]"\n        return brentq(f, lo, hi, xtol=xtol), "interior"\n\n    def s_E(self, E_bar=None, tau=0.0):\n        """equilibrium admissibility threshold: smallest s with B^UE(s) <= 0 (ceiling and floors)."""\n        E_bar = self.E_bar if E_bar is None else E_bar\n        f = lambda s: self.point(s, tau, E_bar)["B"]\n        return self._root(f)\n\n    def s_E_ceiling_only(self, E_bar=None, tau=0.0):\n        E_bar = self.E_bar if E_bar is None else E_bar\n        f = lambda s: self.point(s, tau, E_bar)["E"] - E_bar\n        return self._root(f)\n\n    def s_star(self, E_bar=None, tau=0.0):\n        """structural stationarity under H = s - s_hat: unique root of R(s)=s_hat - s, or a corner."""\n        E_bar = self.E_bar if E_bar is None else E_bar\n        R = lambda s: self.point(s, tau, E_bar)["R"]\n        R0, R1 = R(0.0), R(1.0)\n        if R0 <= 0: return 0.0, "corner_0 (R(0)<=0)"\n        if R1 >= 0: return 1.0, "corner_1 (R(1)>=0)"\n        return brentq(R, 0.0, 1.0, xtol=1e-6), "interior"\n\n    def s_F_upper_price(self, E_bar=None, taus=None, s_tol=1e-3):\n        """Upper bound on the structural feasibility threshold using price-implementable allocations:\n        smallest s such that some tau (acting on routing and demand at fixed s) yields B<=0.\n        Conservative: s_F_true <= returned value."""\n        E_bar = self.E_bar if E_bar is None else E_bar\n        taus = np.concatenate([[0.0], np.logspace(-6, -2, 13)]) if taus is None else taus\n        def minB(s):\n            return min(self.point(s, t, E_bar)["B"] for t in taus)\n        if minB(0.0) <= 0: return 0.0, "at_lower_bound"\n        if minB(1.0) > 0: return np.nan, "none"\n        lo, hi = 0.0, 1.0\n        while hi - lo > s_tol:\n            mid = 0.5 * (lo + hi)\n            if minB(mid) <= 0: hi = mid\n            else: lo = mid\n        return hi, "bisection"\n\n    # ---- sweeps ---------------------------------------------------------------\n    def grid(self, ss, tau=0.0, E_bar=None):\n        return pd.DataFrame([self.point(s, tau, E_bar) for s in ss])\n\n    def ceiling_sweep(self, fracs):\n        rows = []\n        for f in fracs:\n            Eb = f * self.E0\n            sE, kE = self.s_E(Eb); sEc, _ = self.s_E_ceiling_only(Eb); sst, kst = self.s_star(Eb)\n            pt = self.point(sst, 0.0, Eb) if np.isfinite(sst) else {}\n            rows.append(dict(ebar_frac=f, s_E=sE, s_E_kind=kE, s_E_ceiling_only=sEc, s_star=sst, s_star_kind=kst,\n                             gap=(sst - sE) if np.isfinite(sE) else np.nan,\n                             admissible_at_star=bool(pt.get("B", np.inf) <= 0),\n                             s_hat_at_sE=self.point(sE, 0.0, Eb)["s_hat"] if np.isfinite(sE) else np.nan))\n        return pd.DataFrame(rows)\n\n    def regulated_sweep(self, taus, E_bar=None):\n        """tau-regulated coupled equilibrium: for each tau, fixed point s*(tau) with price on all margins."""\n        E_bar = self.E_bar if E_bar is None else E_bar\n        rows = []\n        for t in taus:\n            sst, kind = self.s_star(E_bar, tau=t)\n            pt = self.point(sst, t, E_bar)\n            pt.update(s_star=sst, kind=kind)\n            rows.append(pt)\n        return pd.DataFrame(rows)\n')
print('wrote analysis.py')

In [ ]:
from pathlib import Path
Path('run_experiments.py').write_text('"""First full run: calibration, thresholds, s-grid, ceiling sweep for three configurations."""\nimport sys, time, json, numpy as np, pandas as pd\nfrom dataclasses import replace\nfrom sf_model import Params, CLASSES\nfrom analysis import Model\n\nG0_TARGET, GE_TARGET = -1.1, 4.0          # submitted reduced-form parameters (continuity)\nTAU0 = 1.0e-4                             # $/g  = $100 per tCO2 baseline price (SC-CO2 range $50–190)\nKAPPA = 1.5                               # $/trip annualised EV capital premium (see parameters.md)\n\ndef calibrate(m: Model):\n    """theta from G0 at baseline given kappa, tau0; tau1 from GE."""\n    pt = m.point(0.0, 0.0)\n    Delta0, ebar0 = pt["Delta"], pt["ebar_trip"]\n    arg = Delta0 - KAPPA + TAU0 * ebar0\n    theta = G0_TARGET / arg\n    tau1 = GE_TARGET / (theta * ebar0)\n    return dict(theta=theta, kappa=KAPPA, tau0=TAU0, tau1=tau1, Delta0=Delta0, ebar0=ebar0,\n                check_G0=theta*arg, check_GE=theta*ebar0*tau1)\n\ndef run(name, prm, fracs):\n    t0 = time.time()\n    print(f"\\n=== {name} ===", flush=True)\n    m = Model(prm)\n    cal = calibrate(m)\n    m.prm.theta, m.prm.kappa, m.prm.tau0, m.prm.tau1 = cal["theta"], cal["kappa"], cal["tau0"], cal["tau1"]\n    m._cache.clear()   # s_hat depends on params; equilibria don\'t, but keep it simple\n    print(f"calibration: {json.dumps({k: round(v,6) for k,v in cal.items()})}", flush=True)\n    print(f"E0={m.E0:,.0f}  A0={ {g: round(m.A0[g],1) for g in CLASSES} }  floors={ {g: round(m.floors[g],1) for g in CLASSES} }", flush=True)\n    out = dict(name=name, eps=prm.eps, scheme=prm.class_scheme, E0=m.E0, **{f"A0_{g}": m.A0[g] for g in CLASSES}, **cal)\n    sE, kE = m.s_E();            out.update(s_E=sE, s_E_kind=kE)\n    sEc, _ = m.s_E_ceiling_only(); out.update(s_E_ceiling_only=sEc)\n    sst, kst = m.s_star();       out.update(s_star=sst, s_star_kind=kst)\n    pst = m.point(sst); out.update({f"at_star_{k}": v for k, v in pst.items() if k in ("E","E_frac","A_P","A_C","B","eco_ok","soc_ok","s_hat","tau_eff","q_total")})\n    pE  = m.point(sE) if np.isfinite(sE) else {}\n    out.update(s_hat_at_sE=pE.get("s_hat", np.nan), R_at_sE=pE.get("R", np.nan))\n    print(f"s_E={sE:.4f} ({kE})  s_E_ceiling_only={sEc:.4f}  s*={sst:.4f} ({kst})  gap={sst-sE:+.4f}  "\n          f"E(s*)/E0={pst[\'E_frac\']:.4f}  B(s*)={pst[\'B\']:.3e}  admissible={pst[\'B\']<=0}  s_hat(s_E)={out[\'s_hat_at_sE\']:.4f}", flush=True)\n    grid = m.grid(np.linspace(0, 1, 21)); grid.to_csv(f"results/{name}_sgrid.csv", index=False)\n    # contraction check for partial adjustment (lambda=0.25)\n    Rp = np.gradient(grid["R"].values, grid["s"].values)\n    out.update(Rprime_min=float(Rp.min()), Rprime_max=float(Rp.max()), contraction_modulus_lam025=float(np.max(np.abs(1+0.25*Rp))))\n    print(f"R\'(s) in [{Rp.min():.3f},{Rp.max():.3f}]  contraction modulus (lambda=0.25) = {out[\'contraction_modulus_lam025\']:.3f}", flush=True)\n    sweep = m.ceiling_sweep(fracs); sweep.to_csv(f"results/{name}_ceiling_sweep.csv", index=False)\n    print(sweep.to_string(index=False), flush=True)\n    out.update(runtime_s=time.time()-t0, n_solves=len(m._cache))\n    return out, m\n\nif __name__ == "__main__":\n    fracs = [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]\n    configs = [\n        ("legacy_eps0",     Params(class_scheme="legacy",     eps=0.0)),\n        ("geographic_eps0", Params(class_scheme="geographic", eps=0.0)),\n        ("geographic_eps05",Params(class_scheme="geographic", eps=0.5)),\n    ]\n    summary = []\n    for name, prm in configs:\n        out, _ = run(name, prm, fracs)\n        summary.append(out)\n        pd.DataFrame(summary).to_csv("results/summary_run1.csv", index=False)\n    print("\\nDONE", flush=True)\n')
print('wrote run_experiments.py')

## 3. Baseline model and structural-response calibration

In [ ]:
# Reload local modules cleanly if this notebook is re-run.
for mod in ["sf_model", "analysis", "run_experiments"]:
    if mod in sys.modules:
        del sys.modules[mod]

from sf_model import Params, CLASSES
from analysis import Model
from run_experiments import calibrate

BASE_PARAMS = Params(class_scheme="geographic", eps=0.5)

def setup_model(params=BASE_PARAMS, purchase_stage_charge=True):
    m = Model(params, data_dir=str(DATA_DIR))
    cal = calibrate(m)
    m.prm.theta = cal["theta"]
    m.prm.kappa = cal["kappa"]
    if purchase_stage_charge:
        m.prm.tau0 = cal["tau0"]   # internal name for phi_0
        m.prm.tau1 = cal["tau1"]   # internal name for phi_1
    else:
        m.prm.tau0 = 0.0
        m.prm.tau1 = 0.0
    m._cache.clear()
    return m, cal

m, cal = setup_model()
print(f"E0 = {m.E0:,.3f} g CO2")
print("A0 =", {g: round(m.A0[g], 3) for g in CLASSES})
print("Accessibility floors =", {g: round(m.floors[g], 3) for g in CLASSES})
print("Calibration:")
print(f"  kappa = {cal['kappa']:.3f} $/trip")
print(f"  theta = {cal['theta']:.5f} per $/trip")
print(f"  phi_0 = {cal['tau0']*1e6:.1f} $/tCO2")
print(f"  phi_1 = {cal['tau1']*1e6:.1f} $/tCO2 per unit normalized gap")
print(f"  check G0 = {cal['check_G0']:.6f}; check GE = {cal['check_GE']:.6f}")

The structural response implemented in the code is the vehicle-choice response described in the paper. At a given structural state $s$, the network equilibrium is solved first. The resulting ICE--EV generalized-cost difference and emissions determine the desired EV share $\hat s$. The reduced residual is $R(s)=\hat s-s$, and the stationary structural state satisfies $R(s^*)=0$.

## 4. Table 1 — fixed versus elastic demand

In [ ]:
def threshold_row(eps):
    mm, _ = setup_model(replace(BASE_PARAMS, eps=eps))
    sE, _ = mm.s_E(0.8 * mm.E0)
    ss, _ = mm.s_star(0.8 * mm.E0)
    pt = mm.point(ss, 0.0, 0.8 * mm.E0)
    return {
        "Demand specification": "Fixed demand" if eps == 0 else "Elastic demand",
        "epsilon": eps,
        "s_E": sE,
        "s_star": ss,
        "s_star_minus_s_E": ss - sE,
        "E(s_star)/E0": pt["E_frac"],
    }

table1 = pd.DataFrame([threshold_row(0.0), threshold_row(0.5)])
display(table1.round({"epsilon":1,"s_E":3,"s_star":3,"s_star_minus_s_E":3,"E(s_star)/E0":3}))
table1.to_csv(RESULTS_DIR / "Table_1_thresholds.csv", index=False)

# Manuscript targets (rounding tolerance only)
target1 = [(0.194, 0.220, 0.775), (0.197, 0.221, 0.776)]
for row, target in zip(table1.itertuples(index=False), target1):
    assert abs(row.s_E - target[0]) < 0.003
    assert abs(row.s_star - target[1]) < 0.003
    assert abs(getattr(row, '_5') if False else row[-1] - target[2]) < 0.004
print("Table 1 checks passed.")

### Partial-adjustment stability check

In [ ]:
lambda_adj = 0.25
s_grid_stab = np.linspace(0, 1, 41)
g_stab = m.grid(s_grid_stab, E_bar=0.8*m.E0)
Rprime = np.gradient(g_stab["R"].to_numpy(), g_stab["s"].to_numpy())
modulus = np.max(np.abs(1 + lambda_adj * Rprime))
print(f"max grid value of |1 + lambda R'(s)| = {modulus:.4f}")

# Direct iteration from s0=0.05
s = 0.05
trajectory = [s]
for _ in range(100):
    s = float(np.clip(s + lambda_adj * m.point(s, 0.0, 0.8*m.E0)["R"], 0, 1))
    trajectory.append(s)
    if abs(trajectory[-1] - trajectory[-2]) < 1e-7:
        break
print(f"partial adjustment from s0=0.05 converges numerically to {trajectory[-1]:.6f} in {len(trajectory)-1} steps")

## 5. Instrument-specific price-supported admissibility threshold

In [ ]:
# Exact version of the instrument-specific threshold used in the revised checks.
def price_supported_threshold(mm, ceiling_frac, xtol_s=2e-4):
    Eb = ceiling_frac * mm.E0

    def E(s, t):
        return mm.point(s, t, Eb)["E"]

    def floor_margin(s, t):
        p = mm.point(s, t, Eb)
        return min(p[f"A_{g}"] - mm.floors[g] for g in CLASSES)

    def tau_for_ceiling(s):
        if E(s, 0.0) <= Eb:
            return 0.0
        hi = 6.4e-3
        while E(s, hi) > Eb:
            hi *= 2
            if hi > 0.2:
                return np.inf
        return brentq(lambda t: E(s, t) - Eb, 0.0, hi, xtol=1e-8)

    def tau_for_floor(s):
        if floor_margin(s, 0.0) < 0:
            return -1.0
        if floor_margin(s, 6.4e-3) >= 0:
            return np.inf
        return brentq(lambda t: floor_margin(s, t), 0.0, 6.4e-3, xtol=1e-8)

    def gap(s):
        te = tau_for_ceiling(s)
        ta = tau_for_floor(s)
        if not np.isfinite(te):
            return -1.0
        return ta - te

    sE, _ = mm.s_E(Eb)
    if gap(0.0) >= 0:
        sfp = 0.0
    else:
        sfp = brentq(gap, 0.0, sE, xtol=xtol_s)
    return sfp, sE, tau_for_ceiling(sfp)

rows = []
for eps in [0.5, 0.0]:
    mm, _ = setup_model(replace(BASE_PARAMS, eps=eps))
    for f in ([0.8, 0.7, 0.57] if eps == 0.5 else [0.8]):
        sfp, sE, tp = price_supported_threshold(mm, f)
        rows.append({"epsilon":eps,"ceiling":f,"s_F_price":sfp,"s_E":sE,"price_at_threshold_$/t":tp*1e6})
price_thresholds = pd.DataFrame(rows)
display(price_thresholds.round(3))
price_thresholds.to_csv(RESULTS_DIR / "price_supported_thresholds.csv", index=False)

For elastic demand this is best interpreted as an **instrument-specific price-supported admissibility threshold**: the price changes routing and demand as well as costs. Under fixed demand the price-implementable outcome lies in the original fixed-demand allocation set and therefore provides an upper bound on the fixed-demand structural-feasibility threshold.

## 6. Table 2 — sensitivity to the emissions ceiling

In [ ]:
ceiling_fracs = [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
table2 = m.ceiling_sweep(ceiling_fracs)[["ebar_frac","s_E","s_star","gap","admissible_at_star"]].copy()
table2.columns = ["Ebar/E0","s_E","s_star","s_star_minus_s_E","Stationary state admissible"]
display(table2.round(3))
table2.to_csv(RESULTS_DIR / "Table_2_ceiling_sensitivity.csv", index=False)

expected2 = {
    0.95:(0.049,0.180), 0.90:(0.098,0.183), 0.85:(0.148,0.201), 0.80:(0.197,0.221),
    0.75:(0.247,0.242), 0.70:(0.297,0.263), 0.60:(0.397,0.308), 0.50:(0.497,0.353)
}
for _, r in table2.iterrows():
    a,b = expected2[round(float(r["Ebar/E0"]),2)]
    assert abs(r["s_E"]-a) < 0.003 and abs(r["s_star"]-b) < 0.003
print("Table 2 checks passed.")

## 7. Figure — structural response and admissibility

In [ ]:
# Panel (a): ceiling sweep
sw = table2.sort_values("Ebar/E0")
# Panel (b): structural axis at baseline ceiling
axis_grid = m.grid(np.linspace(0, 1, 41), E_bar=0.8*m.E0)
axis_grid["A_P_frac"] = axis_grid.A_P / m.A0["P"]
axis_grid["A_C_frac"] = axis_grid.A_C / m.A0["C"]
sE08, _ = m.s_E(0.8*m.E0)
ss08, _ = m.s_star(0.8*m.E0)

fig, axs = plt.subplots(1,2,figsize=(10,4))
axs[0].plot(sw["Ebar/E0"], sw["s_E"], marker="o", label=r"$s_E$")
axs[0].plot(sw["Ebar/E0"], sw["s_star"], marker="o", label=r"$s^*$")
d = sw["s_star"].to_numpy()-sw["s_E"].to_numpy(); x = sw["Ebar/E0"].to_numpy()
k = np.where(np.diff(np.sign(d)) != 0)[0]
if len(k):
    i=k[0]; xc=x[i]+(x[i+1]-x[i])*d[i]/(d[i]-d[i+1]); axs[0].axvline(xc,ls=":")
    axs[0].text(xc+0.005,0.44,fr"switch $\approx {xc:.2f}E_0$",rotation=90,va="top")
axs[0].set_xlabel(r"Ceiling $\bar E/E_0$"); axs[0].set_ylabel("EV share"); axs[0].legend(frameon=False)

axs[1].plot(axis_grid.s, axis_grid.E_frac, label=r"$E/E_0$")
axs[1].plot(axis_grid.s, axis_grid.A_P_frac, label=r"$A_P/A_P(0)$")
axs[1].plot(axis_grid.s, axis_grid.A_C_frac, label=r"$A_C/A_C(0)$")
axs[1].axhline(0.8,ls="--"); axs[1].axhline(0.95,ls="--")
axs[1].axvline(sE08,ls=":"); axs[1].axvline(ss08,ls=":")
axs[1].set_xlabel("EV share s"); axs[1].set_ylabel("Fraction of baseline"); axs[1].legend(frameon=False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "Figure_3.pdf", bbox_inches="tight")
fig.savefig(RESULTS_DIR / "Figure_3.png", dpi=220, bbox_inches="tight")
plt.show()

## 8. Pricing with fixed and endogenous structure

In [ ]:
# Tight root tolerances used for prices reported to $0.01/t.
PRICE_XTOL = 1e-8   # $/g = $0.01/t

def stationary_share(mm, tau, Eb):
    R = lambda s: mm.point(s, tau, Eb)["R"]
    r0, r1 = R(0.0), R(1.0)
    if r0 <= 0: return 0.0
    if r1 >= 0: return 1.0
    return brentq(R, 0.0, 1.0, xtol=1e-6)

def price_for_ceiling_fixed_structure(mm, frac):
    Eb = frac*mm.E0
    f = lambda t: mm.point(0.0,t,Eb)["E"]-Eb
    if f(0)<=0: return 0.0
    hi=6.4e-3
    while f(hi)>0:
        hi*=2
        if hi>0.05: return np.nan
    return brentq(f,0,hi,xtol=PRICE_XTOL)

def floor_breach_price(mm, group="P", eps_ceiling_frac=0.8):
    Eb=eps_ceiling_frac*mm.E0
    f=lambda t: mm.point(0.0,t,Eb)[f"A_{group}"]-mm.floors[group]
    return brentq(f,0,6.4e-3,xtol=PRICE_XTOL)

def min_admissible_price_endogenous(mm, frac):
    Eb=frac*mm.E0
    def B(t):
        s=stationary_share(mm,t,Eb)
        return mm.point(s,t,Eb)["B"]
    if B(0)<=0: return 0.0
    return brentq(B,0,6.4e-3,xtol=PRICE_XTOL)

mp, _ = setup_model(BASE_PARAMS)
Pfloor = floor_breach_price(mp,"P")*1e6
Cfloor = floor_breach_price(mp,"C")*1e6

rows=[]
for frac in [0.80,0.70,0.57]:
    Eb=frac*mp.E0
    tf=price_for_ceiling_fixed_structure(mp,frac)
    ta=min_admissible_price_endogenous(mp,frac)
    sa=stationary_share(mp,ta,Eb)
    pa=mp.point(sa,ta,Eb)
    rows.append({
        "Ebar/E0":frac,
        "Fixed: price for ceiling ($/t)":tf*1e6,
        "Fixed: peripheral floor breached ($/t)":Pfloor,
        "Endogenous: admissible price ($/t)":ta*1e6,
        "Endogenous: s_star":sa,
        "Endogenous: A_P/A_P0":pa["A_P"]/mp.A0["P"],
    })
table3=pd.DataFrame(rows)
display(table3.round({"Ebar/E0":2,"Fixed: price for ceiling ($/t)":0,"Fixed: peripheral floor breached ($/t)":0,"Endogenous: admissible price ($/t)":0,"Endogenous: s_star":2,"Endogenous: A_P/A_P0":3}))
table3.to_csv(RESULTS_DIR / "Table_3_pricing.csv", index=False)
print(f"Central accessibility floor at fixed s=0 is breached at about ${Cfloor:.0f}/t.")

# Manuscript values
exp3={0.80:(1899,0,0.22,1.026),0.70:(3430,75,0.29,1.020),0.57:(6602,229,0.41,1.016)}
for _,r in table3.iterrows():
    e=exp3[round(r["Ebar/E0"],2)]
    assert abs(r["Fixed: price for ceiling ($/t)"]-e[0]) < 30
    assert abs(r["Endogenous: admissible price ($/t)"]-e[1]) < 10
    assert abs(r["Endogenous: s_star"]-e[2]) < 0.02
    assert abs(r["Endogenous: A_P/A_P0"]-e[3]) < 0.01
print("Pricing-table checks passed.")

### Fixed-demand floor check

In [ ]:
m_fixed, _ = setup_model(replace(BASE_PARAMS, eps=0.0))
fixed_floor = floor_breach_price(m_fixed,"P")*1e6
print(f"Peripheral floor breach at fixed demand and s=0: ${fixed_floor:.1f}/t (paper: about $151/t)")

### Pricing path at the tight ceiling $0.57E_0$

In [ ]:
Eb57=0.57*mp.E0
for price_t in [0,200,400,800]:
    tau=price_t*1e-6
    ss=stationary_share(mp,tau,Eb57)
    p=mp.point(ss,tau,Eb57)
    print(f"tau=${price_t:>4}/t: s*={ss:.3f}, E/E0={p['E_frac']:.3f}, A_P/A_P0={p['A_P']/mp.A0['P']:.3f}")

## 9. Figure — pricing with and without structural response

In [ ]:
# Fixed structure sweep and endogenous-structure sweep at 0.57 E0.
prices_t=np.array([0,10,25,50,75,100,150,200,229,300,400,600,800,1200,1600,2400,3200,4800,6400,7000],dtype=float)
fixed_rows=[]; endo_rows=[]
for price in prices_t:
    tau=price*1e-6
    pf=mp.point(0.0,tau,Eb57)
    fixed_rows.append({"price":price,"E":pf["E_frac"],"AP":pf["A_P"]/mp.A0["P"],"AC":pf["A_C"]/mp.A0["C"]})
    ss=stationary_share(mp,tau,Eb57); pe=mp.point(ss,tau,Eb57)
    endo_rows.append({"price":price,"E":pe["E_frac"],"AP":pe["A_P"]/mp.A0["P"],"AC":pe["A_C"]/mp.A0["C"],"sstar":ss})
fixed_df=pd.DataFrame(fixed_rows); endo_df=pd.DataFrame(endo_rows)

fig,axs=plt.subplots(1,2,figsize=(11,4))
for ax,df,title in [(axs[0],fixed_df,"Fixed structure, s=0"),(axs[1],endo_df,"Endogenous structure")]:
    ax.plot(df.price,df.E,marker="o",label=r"$E/E_0$")
    ax.plot(df.price,df.AP,marker="o",label=r"$A_P/A_P(0)$")
    ax.plot(df.price,df.AC,marker="o",label=r"$A_C/A_C(0)$")
    if "sstar" in df: ax.plot(df.price,df.sstar,marker="o",ls="--",label=r"$s^*(\tau)$")
    ax.axhline(0.57,ls="--"); ax.axhline(0.95,ls="--")
    ax.set_xscale("symlog",linthresh=10)
    ax.set_xlabel(r"Use-stage emissions price $\tau$ ($/tCO$_2$)")
    ax.set_ylim(0,1.2); ax.set_title(title); ax.legend(frameon=False)
axs[0].set_ylabel("Fraction of baseline / EV share")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "Figure_4.pdf", bbox_inches="tight")
fig.savefig(RESULTS_DIR / "Figure_4.png", dpi=220, bbox_inches="tight")
plt.show()

## 10. Purchase-stage charge removed

In [ ]:
m_no_phi, _ = setup_model(BASE_PARAMS, purchase_stage_charge=False)
rows=[]
for frac in [0.80,0.70,0.57]:
    Eb=frac*m_no_phi.E0
    s0=stationary_share(m_no_phi,0.0,Eb)
    t=min_admissible_price_endogenous(m_no_phi,frac)
    ss=stationary_share(m_no_phi,t,Eb)
    p=m_no_phi.point(ss,t,Eb)
    rows.append({"Ebar/E0":frac,"unpriced s_star":s0,"required price ($/t)":t*1e6,"s_star at price":ss,"A_P/A_P0":p["A_P"]/m_no_phi.A0["P"]})
no_phi=pd.DataFrame(rows)
display(no_phi.round(3))
no_phi.to_csv(RESULTS_DIR / "purchase_stage_charge_removed.csv", index=False)
print("Paper comparison: autonomous s* ≈ 0.18; required prices ≈ $27/t, $163/t and $323/t.")

## 11. Optional robustness table (Appendix)

In [ ]:
def robustness_row(label, params, phi1_mult=1.0):
    mm, cc = setup_model(params)
    if phi1_mult != 1.0:
        mm.prm.tau1 = cc["tau1"]*phi1_mult
        mm._cache.clear()
    sE,_=mm.s_E(0.8*mm.E0); ss,_=mm.s_star(0.8*mm.E0)
    return {"Specification":label,"s_E":sE,"s_star":ss,"gap":ss-sE}

robust=[]
robust.append(robustness_row("Fixed demand, epsilon=0",replace(BASE_PARAMS,eps=0.0)))
robust.append(robustness_row("epsilon=0.3",replace(BASE_PARAMS,eps=0.3)))
robust.append(robustness_row("Central value, epsilon=0.5",BASE_PARAMS))
robust.append(robustness_row("epsilon=0.8",replace(BASE_PARAMS,eps=0.8)))
robust.append(robustness_row("beta_P=beta_C=0.17",replace(BASE_PARAMS,beta={"P":0.17,"C":0.17})))
robust.append(robustness_row("phi_1=0",BASE_PARAMS,phi1_mult=0.0))
robust.append(robustness_row("phi_1 x 2",BASE_PARAMS,phi1_mult=2.0))
robust_df=pd.DataFrame(robust)
display(robust_df.round(3))
robust_df.to_csv(RESULTS_DIR / "Appendix_robustness.csv", index=False)

## 12. Network representation used by the application

In [ ]:
# Recreate a simple version of the paper's network-representation figure.
net=mp.net
eq0=mp.eq(0.0,0.0)
nodes=net.nodes.copy() if hasattr(net,'nodes') else None
# Network stores group_nodes; reload node coordinates directly for plotting.
from sf_model import load_tntp_nodes
nd=load_tntp_nodes(DATA_DIR/'SiouxFalls_node.tntp').sort_values('node').reset_index(drop=True)
coords={int(r.node):(float(r.x),float(r.y)) for r in nd.itertuples()}
fig,axs=plt.subplots(1,2,figsize=(10,4))
for i,(t,h) in enumerate(zip(net.tail+1,net.head+1)):
    x1,y1=coords[int(t)]; x2,y2=coords[int(h)]
    axs[0].plot([x1,x2],[y1,y2],lw=0.5,alpha=0.45)
    width=0.3+2.5*eq0.z[i]/max(eq0.z.max(),1)
    axs[1].plot([x1,x2],[y1,y2],lw=width,alpha=0.5)
P=set(net.group_nodes['P'])
for ax in axs:
    for r in nd.itertuples():
        ax.scatter(r.x,r.y,s=25 if r.node in P else 18,marker='o' if r.node in P else 's')
        ax.text(r.x,r.y,str(int(r.node)),fontsize=6)
    ax.set_aspect('equal'); ax.axis('off')
axs[0].set_title('Origin-based user classes')
axs[1].set_title('Baseline equilibrium flows (line width)')
plt.tight_layout()
fig.savefig(RESULTS_DIR / "Figure_2.pdf", bbox_inches="tight")
fig.savefig(RESULTS_DIR / "Figure_2.png", dpi=220, bbox_inches="tight")
plt.show()

## 13. Reproducibility notes

The main numerical targets in the manuscript are:

- at $\bar E=0.8E_0$, elastic-demand $s_E\approx0.197$ and $s^*\approx0.221$;
- the ordering $s^*-s_E$ changes sign between ceilings $0.80E_0$ and $0.75E_0$, with a switch near $0.76E_0$;
- at fixed structure, the price needed to meet the $0.57E_0$ ceiling is about $\$6{,}602$/t, while the peripheral accessibility floor is breached around $\$205$/t;
- when the same use-stage price also changes vehicle choice, the $0.57E_0$ case becomes admissible at about $\$229$/t with $s^*\approx0.41$;
- removing the purchase-stage charge lowers autonomous adoption to about $0.18$ and raises the role of the use-stage price.

Small numerical differences at the last reported digit can arise from solver warm starts and stopping tolerances. The assertions above are deliberately set at the precision used in the paper rather than at machine precision.

The old repository README should also be updated when this notebook is uploaded: the revised application uses four commodities, elastic demand, geographic classes, equilibrium-admissibility and price-supported thresholds, and a distinct priced coupled equilibrium. The earlier description of a simple fixed point between flows and emissions is no longer sufficient.